# Crosstabs Algorithm

In [ ]:
# The crosstabs algorithm computes the contingency table of two or more categorical
# variables. I suggest to have a brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-crosstab-py/docs/v6-crosstab-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a single (federated-)step algorithm:
#
# 1. Call `partial_crosstab`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the contingency table for
# the entire federated dataset. 
#
# 1. Create a new vantage6 task to execute the *crosstabs* method (central part). This
#    central part will start the tasks `partial_crosstab` (as you can see in the 
#    swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* contingency table from the central part (the main call)
# 4. Retrieve the *local* contingency table from the data stations (the
#    `partial_crosstab` call that was made by the central part)
#

In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [ ]:
# Select the organization IDs that can be selected by the user. These should be the IDs 
# of the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
ORGANIZATION_IDS = [1]

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "crosstab"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [89]

In [ ]:
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

In [ ]:
# The user is able to select two variables to compute the contingency table of.
RESULTS_COL = "sex"
GROUP_COLS = ["fnclcc_grade"] # NOTE: In IDEA4RC we only allow for one group column

In [ ]:
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    # user selected participants
                    "organizations_to_include": ORGANIZATION_IDS 
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))